# 🎬 MiniMax-H3 全模态音视频生成引擎 (A100 / H100 极速纯净版)
> **专为 Antigravity 自动化调度设计的无头 (Headless) 引擎**
- 支持：纯文生视频 (T2V)、首帧生视频 (I2V)、尾帧倒推、首尾双帧强制对齐 (FL2VA)
- 原生全模态：提示词直接渲染 32kHz 原生双声道立体声音效
- 纯净高效：剔除 Gradio 前端和测试冗余，仅保留高可用 API

## 1. 硬件检测与环境初始化

In [ ]:
import os, sys, torch
assert torch.cuda.is_available(), '❌ 未检测到 GPU！请分配 A100 或 H100 运行时。'

gpu_name = torch.cuda.get_device_name(0)
free_mem, total_mem = torch.cuda.mem_get_info('cuda')
vram_gb = total_mem / (1024 ** 3)
vram_limit = max(vram_gb - 2.0, 12.0)

print(f'🚀 GPU 型号: {gpu_name} | 显存: {vram_gb:.1f} GB (可用: {free_mem/(1024**3):.1f} GB) | 预算: {vram_limit:.1f} GB')

## 2. 源码克隆与依赖准备

In [ ]:
if not os.path.exists('/content/DiffSynth-Studio'):
    print('📦 正在克隆 DiffSynth-Studio 源码...')
    !git clone --depth 1 https://github.com/modelscope/DiffSynth-Studio.git /content/DiffSynth-Studio
    !pip install -q -e /content/DiffSynth-Studio
    !pip install -q -U bitsandbytes hf_transfer transformers accelerate safetensors sentencepiece imageio imageio-ffmpeg huggingface_hub av soundfile

if '/content/DiffSynth-Studio' not in sys.path:
    sys.path.insert(0, '/content/DiffSynth-Studio')

os.environ['DIFFSYNTH_DOWNLOAD_SOURCE'] = 'HuggingFace'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('✅ 运行环境与 HuggingFace 极速下载就绪！')

## 3. MiniMax-H3 Pipeline 模型热装载 (NF4 高性能量化)

In [ ]:
import importlib
try:
    import diffsynth.core.quant.backends.bitsandbytes as bnb_backend
    importlib.reload(bnb_backend)
except Exception:
    pass

from diffsynth.pipelines.minimax_h3_audio_video import MiniMaxH3Pipeline, ModelConfig

MODEL_VARIANT = 'NF4'
vram_config = {
    'offload_dtype': torch.bfloat16,
    'offload_device': 'cpu',
    'onload_dtype': torch.bfloat16,
    'onload_device': 'cpu',
    'preparing_dtype': torch.bfloat16,
    'preparing_device': 'cuda',
    'computation_dtype': torch.bfloat16,
    'computation_device': 'cuda',
}

model_configs = [
    ModelConfig(model_id='DiffSynth-Studio/MiniMax-H3-NF4', origin_file_pattern='minimax-h3-fl2va-nf4.safetensors', **vram_config),
    ModelConfig(model_id='DiffSynth-Studio/MiniMax-H3-NF4', origin_file_pattern='minimax-h3-text-encoder-nf4.safetensors', **vram_config),
    ModelConfig(model_id='DiffSynth-Studio/MiniMax-H3-NF4', origin_file_pattern='video_vae_nf4.safetensors', **vram_config),
    ModelConfig(model_id='DiffSynth-Studio/MiniMax-H3-NF4', origin_file_pattern='audio_vae_nf4.safetensors', **vram_config),
]

if 'pipe' not in globals() or globals()['pipe'] is None:
    print('🚀 正在初始化 MiniMaxH3Pipeline (NF4)...')
    pipe = MiniMaxH3Pipeline.from_pretrained(
        torch_dtype=torch.bfloat16,
        device='cuda',
        model_configs=model_configs,
        processor_config=ModelConfig(model_id='MiniMax/MiniMax-H3', origin_file_pattern='FL2VA/processor/'),
        vram_limit=vram_limit,
    )
    print('🎉 MiniMax-H3 引擎已成功挂载至内存与 GPU！')
else:
    print('✅ 检测到现有 Pipeline 实例，无需重复装载。')

## 4. 全模态统一渲染 API (`render_minimax_video`)

In [ ]:
import os, sys, time, datetime, json, shutil
from PIL import Image
from diffsynth.utils.data.audio_video import write_video_audio

def align_to_32(val, min_val=256, max_val=1536):
    val = max(min_val, min(max_val, int(val)))
    return (val + 31) // 32 * 32

def align_frames(target_seconds=5.0, fps=24):
    raw_frames = int(target_seconds * fps)
    n = max(1, round((raw_frames - 5) / 17))
    return 17 * n + 5

def load_as_pil(img_or_path):
    if img_or_path is None:
        return None
    if isinstance(img_or_path, str):
        return Image.open(img_or_path).convert('RGB')
    return img_or_path.convert('RGB')

def render_minimax_video(
    prompt: str,
    first_frame = None,
    end_frame = None,
    width: int = 832,
    height: int = 480,
    duration_sec: float = 5.2,
    num_inference_steps: int = 20,
    seed: int = 42,
    output_dir: str = '/content/outputs'
):
    if 'pipe' not in globals() or globals()['pipe'] is None:
        raise RuntimeError('MiniMaxH3Pipeline 尚未加载，请先执行单元格 3！')

    # 1. 规格校验与对齐
    width = align_to_32(width)
    height = align_to_32(height)
    num_frames = align_frames(duration_sec, fps=24)
    actual_duration = round(num_frames / 24, 2)

    os.makedirs(f'{output_dir}/generated_videos', exist_ok=True)
    os.makedirs(f'{output_dir}/generation_logs', exist_ok=True)
    os.makedirs(f'{output_dir}/previews', exist_ok=True)

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    mp4_path = f'{output_dir}/generated_videos/minimax_{ts}.mp4'
    log_path = f'{output_dir}/generation_logs/minimax_{ts}.json'

    # 2. 图像条件注入 (FL2VA 首尾帧逻辑)
    img_a = load_as_pil(first_frame)
    img_b = load_as_pil(end_frame)
    if img_a is not None:
        img_a = img_a.resize((width, height), Image.LANCZOS)
    if img_b is not None:
        img_b = img_b.resize((width, height), Image.LANCZOS)

    kwargs = {}
    keyframes, keyframe_indices = [], []
    if img_a is not None:
        keyframes.append(img_a)
        keyframe_indices.append(0)
    if img_b is not None:
        keyframes.append(img_b)
        keyframe_indices.append(-1)
    if keyframes:
        kwargs['keyframes'] = keyframes
        kwargs['keyframe_indices'] = keyframe_indices

    mode_name = '首尾帧(FL2VA)' if len(keyframes)==2 else ('首帧驱动(I2V)' if (img_a and not img_b) else ('尾帧倒推' if img_b else '纯文生视频(T2V)'))
    print(f'🎬 [Render Start] 模式: {mode_name} | 分辨率: {width}x{height} | 帧数: {num_frames} ({actual_duration}s) | 步数: {num_inference_steps} | Seed: {seed}')
    print(f'💬 提示词: {prompt}')

    # 3. 真实推理
    t0 = time.time()
    video, audio = pipe(
        prompt=prompt,
        height=height,
        width=width,
        num_frames=num_frames,
        num_inference_steps=num_inference_steps,
        seed=seed,
        **kwargs
    )
    inference_sec = round(time.time() - t0, 2)

    # 4. 音视频写盘与切片提取
    write_video_audio(
        video=video,
        audio=audio,
        output_path=mp4_path,
        fps=24,
        audio_sample_rate=32000,
    )

    preview_paths = {}
    if len(video) > 0:
        first_p = f'{output_dir}/previews/preview_{ts}_first.png'
        mid_p = f'{output_dir}/previews/preview_{ts}_mid.png'
        last_p = f'{output_dir}/previews/preview_{ts}_last.png'
        video[0].save(first_p)
        video[len(video)//2].save(mid_p)
        video[-1].save(last_p)
        preview_paths = {'first': first_p, 'mid': mid_p, 'last': last_p}

    metadata = {
        'timestamp': ts,
        'prompt': prompt,
        'mode': mode_name,
        'has_first_frame': img_a is not None,
        'has_end_frame': img_b is not None,
        'keyframe_indices': keyframe_indices,
        'width': width,
        'height': height,
        'num_frames': num_frames,
        'duration_sec': actual_duration,
        'num_inference_steps': num_inference_steps,
        'seed': seed,
        'inference_time_sec': inference_sec,
        'video_path': mp4_path,
        'previews': preview_paths,
        'audio_sample_rate': 32000,
        'audio_channels': 2
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f'✨ [Render Complete] 耗时: {inference_sec}s | 文件大小: {os.path.getsize(mp4_path)/(1024*1024):.2f} MB')
    return metadata

print('✅ render_minimax_video() API 就绪，可随时接收动态参数调用！')